<a href="https://colab.research.google.com/github/ravidu-hevaganinge/Broiler-Vision/blob/main/9_5_25_Chicken_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
import os
import cv2
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt


# Mount Google Drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install tensorboard

In [3]:
#functions and libraries
from skimage.feature import local_binary_pattern

import torch
from torchvision import datasets, transforms, models
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
import pandas as pd

from torch.utils.tensorboard import SummaryWriter
import tensorflow as tf
import tensorboard as tb
tf.io.gfile = tb.compat.tensorflow_stub.io.gfile

import torch
from torch.utils.data import Dataset, DataLoader
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import torch.nn as nn

def enhance_meat_texture(data_path,output_file,radius):

  radius = radius
  n_points = 8 * radius

  # Get all files in directory
  file_names = os.listdir(data_path)
  # Supported image extensions
  image_extensions = ('.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.tif', '.webp')

  for file_name in file_names:
      # Skip if not an image file
      if not file_name.lower().endswith(image_extensions):
          continue

      try:
          # Construct full file path
          image_path = os.path.join(data_path, file_name)

          # Read and process the image
          img_bgr = cv2.imread(image_path)
          if img_bgr is None:
              print(f"Could not read image: {file_name}")
              continue

          img_hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)

          # Process each channel
          h = local_binary_pattern(img_hsv[:,:,0], n_points, radius, method='uniform')
          s = local_binary_pattern(img_hsv[:,:,1], n_points, radius, method='uniform')
          v = local_binary_pattern(img_hsv[:,:,2], n_points, radius, method='uniform')

          # Normalize each channel
          h_norm = ((h - h.min()) * (255.0 / (h.max() - h.min()))).astype(np.uint8)
          s_norm = ((s - s.min()) * (255.0 / (s.max() - s.min()))).astype(np.uint8)
          v_norm = ((v - v.min()) * (255.0 / (v.max() - v.min()))).astype(np.uint8)

          # Merge channels
          img_proc_color = cv2.merge([h_norm, s_norm, v_norm])

          # Save processed image
          output_path = os.path.join(output_file, "processed_" + file_name)
          cv2.imwrite(output_path, img_proc_color)
          #print(f"Processed: {file_name}")

      except Exception as e:
          print(f"Error processing {file_name}: {str(e)}")

  print("Processing complete!")

In [4]:
# Define parameters
radius = 3
root = '/content/drive/MyDrive/Krab Lab/Summer 2025'
# Input and output paths
output_file = root + "/Chicken Project/Data/080725/test"
label_path = root + '/Chicken Project/Data/080725/Chicken_breast.csv'
img_dir = root + '/Chicken Project/Data/080725/proccesed all'
# enhance_meat_texture(data_path,output_file,radius)


In [5]:
labels_df = pd.read_csv(label_path)
labels_df.head()

,ID,Rating,fold
0,breast_1,woody,train
1,breast_2,woody,train
2,breast_3,woody,train
3,breast_4,woody,train
4,breast_5,normal,train


In [6]:
labels_df = pd.read_csv(label_path)
model_pipeline = transforms.Compose([transforms.CenterCrop([1000,800]),transforms.ToTensor(),transforms.Normalize(mean = [0,0,0],std= [1,1,1])])

# KEY regarding for labels
# 0 - normal
# 1 - spaghetti
# 2 - woody

class ChickenBreastDataset(Dataset):
    def __init__(self, data_dir, labels_df, transform=None):
        self.data_dir = data_dir
        self.labels_df = labels_df
        self.transform = transform
        class_names = sorted(labels_df['Rating'].unique())
        self.stoi = {s:i for i,s in enumerate(class_names)}
        print(self.stoi)

    def __len__(self):
        return len(self.labels_df)

    def __getitem__(self, idx):
        img_name = os.path.join(self.data_dir, 'Copy of processed_' + self.labels_df.iloc[idx, 0] + '.png')
        image = Image.open(img_name).convert('RGB')
        label = self.labels_df.iloc[idx, 1]


        if self.transform:
            image = self.transform(image)
        numerical_label = self.stoi[label]
        label = torch.tensor(numerical_label)
        label = F.one_hot(label, num_classes=3).float()
        return image, label



In [7]:
from sklearn.model_selection import StratifiedKFold
# Set up StratifiedKFold
n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
train_index, test_index = next(skf.split(labels_df.to_numpy(),labels_df['Rating'].to_numpy()))
train_df = labels_df.iloc[train_index].copy()
test_df = labels_df.iloc[test_index].copy()

model_pipeline = transforms.Compose([transforms.CenterCrop([1000,800]),transforms.ToTensor(),transforms.Normalize(mean = [0,0,0],std= [1,1,1])])

train_dataset = ChickenBreastDataset(img_dir, train_df, model_pipeline)
test_dataset = ChickenBreastDataset(img_dir, test_df, model_pipeline)

#Create dataloaders
batch_size = 5
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=True)

{'normal': 0, 'spaghetti': 1, 'woody': 2}
{'normal': 0, 'spaghetti': 1, 'woody': 2}


In [8]:
class ChickenClassifier(nn.Module):
    def __init__(self, num_classes):
        super(ChickenClassifier, self).__init__()
        feature_ext = models.resnet50(pretrained=True)
        feature_ext.fc = nn.Identity()
        #pretrained feature extractor is frozen
        for param in feature_ext.parameters():
            param.requires_grad = False


        self.feature_ext = feature_ext
        self.classifier = torch.nn.Linear(2048, num_classes)

    def forward(self, x):
        x = self.feature_ext(x)
        x = F.relu(x)
        x = self.classifier(x)
        x = F.softmax(x, dim=1)
        return x

In [20]:
X_i,y_i = next(iter(train_loader))
chicken_model = ChickenClassifier(3)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
chicken_model.to(device)

X_i = X_i.to(device, non_blocking=True)
y_i = y_i.to(device, non_blocking=True)

# need to setup a loss function and an optimization scheme prior to using the neural network.
loss_fn = torch.nn.CrossEntropyLoss()
loss_fn.to(device)
# using Adam optimizer
optimizer = torch.optim.Adam(chicken_model.parameters(), lr=0.001)

chicken_model.train()
optimizer.zero_grad()
print(X_i.shape)
ypi = chicken_model(X_i)
print(ypi.shape)

loss = loss_fn(ypi,y_i)
print(loss)
loss.backward()
optimizer.step()

# because my dataset is so small, should I use the test data as validation data or should I implement a cross validation scheme with the training data?
X_val,y_val = next(iter(test_loader))


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


torch.Size([5, 3, 1000, 800])
torch.Size([5, 3])
tensor(0.9612, device='cuda:0', grad_fn=<DivBackward1>)
